In [ ]:
import os
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import squareform
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.gridspec as gridspec
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import partial_dependence
from collections import defaultdict
import shap
import pickle
import re
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

X_PATH = r'J:\DATA\project\data\All_variables_merged.nc'
Y1_PATH = r'J:\DATA\project\trend\2011-2020_Spring_trend.nc'
MASK_STEP1_PATH = r'J:\DATA\project\mask\significant_mask_p005_Spring_maxcorr.nc'
MASK_STEP23_PATH = r'J:\DATA\project\climate\Temperate.nc'
OUTPUT_DIR = r'J:\DATA\project\output\shap_results'

INITIAL_VARS = [
    'VPD_mean_Spring', 'VPD_trend_Spring', 'VPD_cv_Spring',
    'SRAD_mean_Spring', 'SRAD_trend_Spring', 'SRAD_cv_Spring',
    'LAI_mean_Spring', 'LAI_trend_Spring', 'LAI_cv_Spring',
    'CO2_mean_Spring', 'CO2_trend_Spring', 'CO2_cv_Spring',
    'PRE_mean_Spring', 'PRE_trend_Spring', 'PRE_cv_Spring',
    'TMP_mean_Spring', 'TMP_trend_Spring', 'TMP_cv_Spring',
    'VPD_Y_Mean', 'VPD_Y_Trend', 'VPD_Y_CV',
    'SRAD_Y_Mean', 'SRAD_Y_Trend', 'SRAD_Y_CV',
    'LAI_Y_Mean', 'LAI_Y_Trend', 'LAI_Y_CV',
    'CO2_Y_Mean', 'CO2_Y_Trend', 'CO2_Y_CV',
    'PRE_Y_Mean', 'PRE_Y_Trend', 'PRE_Y_CV',
    'TMP_Y_Mean', 'TMP_Y_Trend', 'TMP_Y_CV'
]

VAR_LABELS = {
    'VPD_mean_Spring': 'VPD Mean', 'VPD_trend_Spring': 'VPD Trend', 'VPD_cv_Spring': 'VPD CV',
    'SRAD_mean_Spring': 'SRAD Mean', 'SRAD_trend_Spring': 'SRAD Trend', 'SRAD_cv_Spring': 'SRAD CV',
    'LAI_mean_Spring': 'LAI Mean', 'LAI_trend_Spring': 'LAI Trend', 'LAI_cv_Spring': 'LAI CV',
    'CO2_mean_Spring': 'CO$_2$ Mean', 'CO2_trend_Spring': 'CO$_2$ Trend', 'CO2_cv_Spring': 'CO$_2$ CV',
    'PRE_mean_Spring': 'PRE Mean', 'PRE_trend_Spring': 'PRE Trend', 'PRE_cv_Spring': 'PRE CV',
    'TMP_mean_Spring': 'TEM Mean', 'TMP_trend_Spring': 'TEM Trend', 'TMP_cv_Spring': 'TEM CV',
    'VPD_Y_Mean': 'VPD Y_Mean', 'VPD_Y_Trend': 'VPD Y_Trend', 'VPD_Y_CV': 'VPD Y_CV',
    'SRAD_Y_Mean': 'SRAD Y_Mean', 'SRAD_Y_Trend': 'SRAD Y_Trend', 'SRAD_Y_CV': 'SRAD Y_CV',
    'LAI_Y_Mean': 'LAI Y_Mean', 'LAI_Y_Trend': 'LAI Y_Trend', 'LAI_Y_CV': 'LAI Y_CV',
    'CO2_Y_Mean': 'CO$_2$ Y_Mean', 'CO2_Y_Trend': 'CO$_2$ Y_Trend', 'CO2_Y_CV': 'CO$_2$ Y_CV',
    'PRE_Y_Mean': 'PRE Y_Mean', 'PRE_Y_Trend': 'PRE Y_Trend', 'PRE_Y_CV': 'PRE Y_CV',
    'TMP_Y_Mean': 'TEM Y_Mean', 'TMP_Y_Trend': 'TEM Y_Trend', 'TMP_Y_CV': 'TEM Y_CV'
}

Y_VAR = 'theil_sen_slope'
Y_LABEL = 'TS$_d$ Slope'
CLUSTER_THRESHOLD = 0.7
P_THRESHOLD = 0.05


def ensure_output_dir(path):
    os.makedirs(path, exist_ok=True)


def unify_coords(da):
    coord_map = {}
    for c in da.coords:
        if c.lower() in ['latitude', 'lat']:
            coord_map[c] = 'lat'
        if c.lower() in ['longitude', 'lon']:
            coord_map[c] = 'lon'
    return da.rename(coord_map)


def squeeze_to_latlon(da):
    for d in list(da.dims):
        if d not in ['lat', 'lon']:
            da = da.isel({d: 0})
    return da


def fix_co2_label(label):
    label = label.replace('CO₂', 'CO2')
    label = re.sub(r'CO2', r'CO$_2$', label)
    return label


def build_significant_mask(y_da, sig_mask_path, sig_var='significant_mask'):
    sig_ds = xr.open_dataset(sig_mask_path)
    sig_mask = unify_coords(sig_ds[sig_var])
    sig_mask = squeeze_to_latlon(sig_mask)
    target_lat = y_da['lat']
    target_lon = y_da['lon']
    sig_mask_interp = sig_mask.interp(lat=target_lat, lon=target_lon, method='nearest')
    final_mask = (sig_mask_interp == 1).astype(int)
    return final_mask


def build_intersection_mask(y_da, sig_mask_path, climate_mask_path,
                            sig_var='significant_mask', climate_var='mask'):
    sig_ds = xr.open_dataset(sig_mask_path)
    climate_ds = xr.open_dataset(climate_mask_path)

    sig_mask = unify_coords(sig_ds[sig_var])
    sig_mask = squeeze_to_latlon(sig_mask)

    climate_mask = unify_coords(climate_ds[climate_var])
    climate_mask = squeeze_to_latlon(climate_mask)

    target_lat = y_da['lat']
    target_lon = y_da['lon']

    sig_mask_interp = sig_mask.interp(lat=target_lat, lon=target_lon, method='nearest')
    climate_mask_interp = climate_mask.interp(lat=target_lat, lon=target_lon, method='nearest')

    final_mask = ((sig_mask_interp == 1) & (climate_mask_interp == 1)).astype(int)
    return final_mask


def step1_hexbin_plot(x_vars, output_suffix=''):
    X_ds = xr.open_dataset(X_PATH)
    Y_ds = xr.open_dataset(Y1_PATH)

    Y = unify_coords(Y_ds[Y_VAR])
    Y = squeeze_to_latlon(Y)
    target_lat = Y['lat']
    target_lon = Y['lon']

    final_mask = build_significant_mask(
        y_da=Y,
        sig_mask_path=MASK_STEP1_PATH,
        sig_var='significant_mask'
    )

    colors = ['#440154', '#31688e', '#35b779', '#fde725']
    cmap = LinearSegmentedColormap.from_list('viridis_custom', colors, N=256)

    plt.rcParams['font.family'] = 'DejaVu Sans'
    plt.rcParams['font.size'] = 10

    n_vars = len(x_vars)
    n_cols = 4
    n_rows = (n_vars + n_cols - 1) // n_cols

    fig = plt.figure(figsize=(n_cols * 4, n_rows * 3))
    fig.patch.set_facecolor('white')

    gs = gridspec.GridSpec(
        n_rows, n_cols, figure=fig,
        hspace=0.35, wspace=0.25,
        left=0.06, right=0.92, top=0.95, bottom=0.05
    )

    hb_for_cbar = None
    significant_vars = []

    for idx, x_var in enumerate(x_vars):
        if x_var not in X_ds:
            continue

        row = idx // n_cols
        col = idx % n_cols
        ax = fig.add_subplot(gs[row, col])

        X = unify_coords(X_ds[x_var])
        X = squeeze_to_latlon(X)
        X_interp = X.interp(lat=target_lat, lon=target_lon, method='nearest')
        X_masked = X_interp.where(final_mask == 1)
        Y_masked = Y.where(final_mask == 1)

        if 'time' in X_masked.dims and 'time' in Y_masked.dims:
            nlat = X_masked.sizes['lat']
            nlon = X_masked.sizes['lon']
            r_mat = np.full((nlat, nlon), np.nan)
            p_mat = np.full((nlat, nlon), np.nan)

            for i in range(nlat):
                for j in range(nlon):
                    x = X_masked[:, i, j].values
                    y = Y_masked[:, i, j].values
                    valid = ~np.isnan(x) & ~np.isnan(y)
                    if np.sum(valid) > 2:
                        r, p = stats.spearmanr(x[valid], y[valid])
                        r_mat[i, j] = r
                        p_mat[i, j] = p

            x_mean = np.nanmean(X_masked.values, axis=0)
            y_mean = np.nanmean(Y_masked.values, axis=0)
            valid_mask = (p_mat < 0.05) & (~np.isnan(x_mean)) & (~np.isnan(y_mean))
            X_valid = x_mean[valid_mask]
            Y_valid = y_mean[valid_mask]
        else:
            X_flat = X_masked.values.flatten()
            Y_flat = Y_masked.values.flatten()
            valid = ~np.isnan(X_flat) & ~np.isnan(Y_flat)
            X_valid = X_flat[valid]
            Y_valid = Y_flat[valid]

        if X_valid.size == 0:
            ax.set_visible(False)
            continue

        slope, intercept, _, _, _ = stats.linregress(X_valid, Y_valid)
        x_vals = np.array([np.nanmin(X_valid), np.nanmax(X_valid)])
        x_range = np.linspace(x_vals[0], x_vals[1], 100)
        y_pred = slope * x_range + intercept
        residuals = Y_valid - (slope * X_valid + intercept)
        mse = np.mean(residuals**2)
        std_err_pred = np.sqrt(mse)
        ci = 1.96 * std_err_pred

        ax.fill_between(x_range, y_pred - ci, y_pred + ci, color="#2c77e7", alpha=0.18, zorder=1)

        hb = ax.hexbin(X_valid, Y_valid, gridsize=50, cmap=cmap, mincnt=1,
                       linewidths=0.1, alpha=0.85, zorder=2)
        if hb_for_cbar is None:
            hb_for_cbar = hb

        ax.plot(x_vals, slope * x_vals + intercept, color='#d62728', lw=2.5, alpha=0.95, zorder=3)

        r, p = stats.spearmanr(X_valid, Y_valid)
        if p < P_THRESHOLD:
            significant_vars.append(x_var)

        if p < 0.001:
            p_str = "p < 0.001"
        elif p < 0.01:
            p_str = "p < 0.01"
        elif p < 0.05:
            p_str = "p < 0.05"
        else:
            p_str = f"p = {p:.2f}"

        text_str = f"r = {r:.2f}\n{p_str}"
        bbox_props = dict(boxstyle="round,pad=0.3", facecolor='white',
                          edgecolor='gray', alpha=0.9, linewidth=1)
        ax.text(0.05, 0.95, text_str, transform=ax.transAxes,
                fontsize=10, verticalalignment='top', fontweight='bold',
                bbox=bbox_props, color='#2c3e50')

        ax.set_title(VAR_LABELS.get(x_var, x_var), fontsize=12, fontweight='bold', pad=15, color="#030303")
        ax.grid(True, alpha=0.25, linestyle='--', linewidth=0.8)
        ax.set_axisbelow(True)

        if col == 0:
            ax.set_ylabel(Y_LABEL, fontsize=11, fontweight='bold')
        else:
            ax.set_ylabel('')

        ax.tick_params(axis='both', which='major', labelsize=9, colors='#2c3e50', width=1.2, length=4)
        for spine in ax.spines.values():
            spine.set_color('#2c3e50')
            spine.set_linewidth(1.2)

    total_positions = n_rows * n_cols
    for idx in range(n_vars, total_positions):
        row = idx // n_cols
        col = idx % n_cols
        ax = fig.add_subplot(gs[row, col])
        ax.set_visible(False)

    if hb_for_cbar is not None:
        cbar_ax = fig.add_axes([0.93, 0.15, 0.015, 0.7])
        cbar = fig.colorbar(hb_for_cbar, cax=cbar_ax, orientation='vertical')
        cbar.set_label('Point Density', fontsize=12, fontweight='bold', color='#2c3e50')
        cbar.ax.tick_params(labelsize=10, colors='#2c3e50', width=1.2, length=4)
        cbar.outline.set_color('#2c3e50')
        cbar.outline.set_linewidth(1.2)

    fig.suptitle('Correlation Analysis: Environmental Variables vs TS Slope',
                 fontsize=16, fontweight='bold', y=0.98, color='#2c3e50')
    fig.text(
        0.5, 0.02,
        'Red lines show linear regression; blue shading is 95% confidence interval. Hexagonal bins represent point density.',
        ha='center', fontsize=10, style='italic', color='#7f8c8d'
    )

    output_filename = f'{OUTPUT_DIR}\\hexbin_all_spring_rd_{output_suffix}.png'
    fig.savefig(output_filename, dpi=600, bbox_inches='tight', pad_inches=0.05)
    plt.show()

    return significant_vars


def step2_hierarchical_clustering(x_vars, output_suffix=''):
    X_ds = xr.open_dataset(X_PATH)
    Y_ds = xr.open_dataset(Y1_PATH)

    Y = unify_coords(Y_ds[Y_VAR])
    Y = squeeze_to_latlon(Y)
    target_lat = Y['lat']
    target_lon = Y['lon']

    final_mask = build_intersection_mask(
        y_da=Y,
        sig_mask_path=MASK_STEP1_PATH,
        climate_mask_path=MASK_STEP23_PATH,
        sig_var='significant_mask',
        climate_var='mask'
    )

    all_data = []
    valid_vars = []

    for x_var in x_vars:
        if x_var not in X_ds:
            continue

        X = unify_coords(X_ds[x_var])
        X = squeeze_to_latlon(X)
        X_interp = X.interp(lat=target_lat, lon=target_lon, method='nearest')
        X_masked = X_interp.where(final_mask == 1)
        all_data.append(X_masked.values.flatten())
        valid_vars.append(x_var)

    if len(all_data) == 0:
        raise ValueError("No variables available for clustering.")

    data_mat = np.stack(all_data, axis=1)
    df_data = pd.DataFrame(data_mat, columns=valid_vars).dropna(how='any')

    if len(df_data) < 3:
        raise ValueError("Too few valid samples for clustering.")

    corr = df_data.corr(method='spearman')
    distance = 1 - np.abs(corr)
    np.fill_diagonal(distance.values, 0)

    dists = squareform(distance.values, checks=False)
    linkage = sch.linkage(dists, method='average')

    distance_threshold = 1 - CLUSTER_THRESHOLD
    cluster_labels = sch.fcluster(linkage, distance_threshold, criterion='distance')

    clusters = defaultdict(list)
    for i, label in enumerate(cluster_labels):
        clusters[label].append((valid_vars[i], i))

    selected_vars = []
    selected_indices = []

    for _, vars_in_cluster in clusters.items():
        if len(vars_in_cluster) == 1:
            var_name, var_idx = vars_in_cluster[0]
            selected_vars.append(var_name)
            selected_indices.append(var_idx)
        else:
            var_indices_in_cluster = [v[1] for v in vars_in_cluster]
            max_mean_corr = -1
            best_var = None
            best_idx = None

            for var_name, var_idx in vars_in_cluster:
                other_indices = [idx for idx in var_indices_in_cluster if idx != var_idx]
                if other_indices:
                    mean_corr = np.mean([abs(corr.iloc[var_idx, other_idx]) for other_idx in other_indices])
                    if mean_corr > max_mean_corr:
                        max_mean_corr = mean_corr
                        best_var = var_name
                        best_idx = var_idx
                else:
                    best_var = var_name
                    best_idx = var_idx

            selected_vars.append(best_var)
            selected_indices.append(best_idx)

    plt.figure(figsize=(12, 6))
    sch.dendrogram(linkage, labels=[VAR_LABELS.get(v, v) for v in valid_vars],
                   leaf_rotation=45, leaf_font_size=10)
    plt.axhline(y=distance_threshold, color='red', linestyle='--', alpha=0.7,
                label=f'Threshold {CLUSTER_THRESHOLD}')
    plt.title('Hierarchical Clustering Dendrogram', fontsize=14)
    plt.ylabel('Distance (1 - |correlation|)', fontsize=12)
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 10))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', center=0,
                xticklabels=[VAR_LABELS.get(v, v) for v in valid_vars],
                yticklabels=[VAR_LABELS.get(v, v) for v in valid_vars],
                cbar_kws={'label': 'Spearman Correlation'})
    plt.title(f'Original Correlation Matrix ({len(valid_vars)} variables)', fontsize=14)
    plt.tight_layout()
    plt.show()

    selected_corr = corr.iloc[selected_indices, selected_indices]
    selected_labels = [VAR_LABELS.get(v, v) for v in selected_vars]

    plt.figure(figsize=(10, 8))
    sns.heatmap(selected_corr, annot=True, fmt=".2f", cmap='coolwarm', center=0,
                xticklabels=selected_labels, yticklabels=selected_labels,
                cbar_kws={'label': 'Spearman Correlation'})
    plt.title(f'Selected Correlation Matrix ({len(selected_vars)} variables)', fontsize=14)
    plt.tight_layout()
    plt.show()

    g = sns.clustermap(corr, method='average', cmap='coolwarm', center=0,
                       linewidths=.5, figsize=(12, 12),
                       xticklabels=[VAR_LABELS.get(v, v) for v in valid_vars],
                       yticklabels=[VAR_LABELS.get(v, v) for v in valid_vars],
                       cbar_pos=(0.02, 0.02, 0.03, 0.15))

    output_filename = f'{OUTPUT_DIR}\\cluster_heatmap_all_spring_rd_{output_suffix}.png'
    g.savefig(output_filename, dpi=600, bbox_inches='tight', pad_inches=0.05)
    plt.show()

    output_file = f'{OUTPUT_DIR}\\selected_variables_spring_rd_{output_suffix}.txt'
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(f'Selection results (threshold={CLUSTER_THRESHOLD})\n')
        f.write('Mask: significant_mask ∩ temperate mask\n')
        f.write(f'Initial variables: {len(valid_vars)}\n')
        f.write(f'Selected variables: {len(selected_vars)}\n')
        f.write(f'Reduction ratio: {(1-len(selected_vars)/len(valid_vars))*100:.1f}%\n\n')
        f.write('Selected variable list:\n')
        for i, var in enumerate(selected_vars, 1):
            f.write(f'{i:2d}. {var}\n')

    return selected_vars


def step3_shap_analysis_and_plot(x_vars, output_suffix=''):
    X_ds = xr.open_dataset(X_PATH)
    Y1_ds = xr.open_dataset(Y1_PATH)

    Y1 = unify_coords(Y1_ds[Y_VAR])
    Y1 = squeeze_to_latlon(Y1)
    target_lat = Y1['lat']
    target_lon = Y1['lon']

    final_mask = build_intersection_mask(
        y_da=Y1,
        sig_mask_path=MASK_STEP1_PATH,
        climate_mask_path=MASK_STEP23_PATH,
        sig_var='significant_mask',
        climate_var='mask'
    )

    all_data = []
    valid_vars = []

    for v in x_vars:
        if v not in X_ds:
            continue

        X = unify_coords(X_ds[v])
        X = squeeze_to_latlon(X)
        X_interp = X.interp(lat=target_lat, lon=target_lon, method='nearest')
        X_masked = X_interp.where(final_mask == 1)
        all_data.append(X_masked.values.flatten())
        valid_vars.append(v)

    if len(all_data) == 0:
        raise ValueError("No variables available for SHAP analysis.")

    X_mat = np.stack(all_data, axis=1)
    df_X = pd.DataFrame(X_mat, columns=valid_vars)

    Y1_masked = Y1.where(final_mask == 1).values.flatten()
    df_X['TSd'] = Y1_masked
    df_X = df_X.dropna(how='any')

    X = df_X[valid_vars].values
    y1 = df_X['TSd'].values

    if len(y1) < 10:
        raise ValueError("Too few valid samples for random forest training.")

    rf = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
    rf.fit(X, y1)
    importances = rf.feature_importances_
    r2_score = rf.score(X, y1)

    n_top = min(12, len(valid_vars))
    idx = np.argsort(importances)[::-1][:n_top]
    sel_vars = [valid_vars[i] for i in idx]
    sel_labels = [VAR_LABELS.get(v, v) for v in sel_vars]
    sel_importances = importances[idx]

    sample_size = min(800, len(X))
    np.random.seed(42)
    sample_idx = np.random.choice(len(X), sample_size, replace=False)
    X_sample = X[sample_idx]

    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_sample)

    shap_selected = shap_values[:, idx]
    X_selected = X_sample[:, idx]

    pdp_results = []
    for i in range(n_top):
        pdp_result = partial_dependence(rf, X, features=[idx[i]], grid_resolution=50)
        pdp_results.append(pdp_result)

    climate_vars = [v for v in sel_vars if any(k in v for k in ['VPD', 'SRAD', 'PRE', 'TMP'])]
    plant_vars = [v for v in sel_vars if 'LAI' in v]
    env_vars = [v for v in sel_vars if 'CO2' in v]

    var_categories = {}
    for v in climate_vars:
        var_categories[v] = 'climate'
    for v in plant_vars:
        var_categories[v] = 'plant'
    for v in env_vars:
        var_categories[v] = 'environment'

    category_colors = {
        'climate': '#a8dadc',
        'plant': '#c1e7a4',
        'environment': '#f4a6a6'
    }
    colors = [category_colors.get(var_categories.get(v, 'other'), '#cccccc') for v in sel_vars]

    results = {
        'analysis_info': {
            'analysis_type': 'filtered_variables_shap_intersection_mask',
            'n_input_vars': len(valid_vars),
            'n_top_vars': n_top,
            'sample_size': sample_size,
            'total_samples': len(y1),
            'r2_score': r2_score,
            'mask_step1': MASK_STEP1_PATH,
            'mask_step23': MASK_STEP23_PATH,
            'final_mask': 'significant_mask_intersection_temperate_mask',
            'calculation_time': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
        },
        'input_variables': {
            'variables': valid_vars,
            'labels': [VAR_LABELS.get(v, v) for v in valid_vars],
            'source': 'hierarchical_clustering_filtered'
        },
        'importance_ranking': {
            'all_variables': [],
            'top_variables': []
        },
        'shap_data': {
            'shap_values': shap_selected,
            'feature_values': X_selected,
            'variable_names': sel_vars,
            'variable_labels': sel_labels,
            'sample_indices': sample_idx
        },
        'pdp_data': {
            'pdp_results': pdp_results,
            'variable_names': sel_vars
        },
        'display_info': {
            'colors': colors,
            'categories': var_categories,
            'top_importances': sel_importances
        },
        'model_info': {
            'model_type': 'RandomForestRegressor',
            'n_estimators': 300,
            'random_state': 42,
            'feature_names': valid_vars,
            'importance_sum': float(np.sum(importances))
        }
    }

    all_ranks = np.argsort(importances)[::-1]
    for i, var in enumerate(valid_vars):
        results['importance_ranking']['all_variables'].append({
            'rank': int(np.where(all_ranks == i)[0][0]) + 1,
            'variable': var,
            'label': VAR_LABELS.get(var, var),
            'importance': importances[i]
        })

    results['importance_ranking']['all_variables'].sort(key=lambda x: x['rank'])
    results['importance_ranking']['top_variables'] = results['importance_ranking']['all_variables'][:n_top]

    save_path_pkl = f'{OUTPUT_DIR}\\shap_results_filtered_{output_suffix}_spring_rd.pkl'
    with open(save_path_pkl, 'wb') as f:
        pickle.dump(results, f)

    save_path_shap = f'{OUTPUT_DIR}\\shap_values_matrix_{output_suffix}_spring_rd.npy'
    np.save(save_path_shap, shap_selected)

    save_path_csv = f'{OUTPUT_DIR}\\importance_ranking_{output_suffix}_spring_rd.csv'
    df_importance = pd.DataFrame(results['importance_ranking']['all_variables'])
    df_importance.to_csv(save_path_csv, index=False, encoding='utf-8')

    save_path_txt = f'{OUTPUT_DIR}\\shap_analysis_report_{output_suffix}_spring_rd.txt'
    with open(save_path_txt, 'w', encoding='utf-8') as f:
        f.write('=' * 80 + '\n')
        f.write('SHAP importance analysis report\n')
        f.write('=' * 80 + '\n')
        f.write(f"Time: {results['analysis_info']['calculation_time']}\n")
        f.write(f"Mask step1: {MASK_STEP1_PATH}\n")
        f.write(f"Mask step2/3: intersection with {MASK_STEP23_PATH}\n")
        f.write(f"Input variables: {len(valid_vars)}\n")
        f.write(f"Top variables: {n_top}\n")
        f.write(f"Samples: {len(y1)} (SHAP sample: {sample_size})\n")
        f.write(f"R2: {r2_score:.4f}\n")
        f.write(f"Importance sum: {np.sum(importances):.10f}\n\n")
        f.write('Full importance ranking:\n')
        f.write('-' * 50 + '\n')
        for item in results['importance_ranking']['all_variables']:
            f.write(f"{item['rank']:2d}. {item['label']:20s}: {item['importance']:.6f}\n")

    save_path_excel = f'{OUTPUT_DIR}\\shap_data_{output_suffix}_spring_rd.xlsx'
    try:
        with pd.ExcelWriter(save_path_excel, engine='openpyxl') as writer:
            df_importance.to_excel(writer, sheet_name='Importance_Ranking', index=False)

            shap_summary = pd.DataFrame({
                'Variable': sel_labels,
                'Mean_SHAP_abs': np.mean(np.abs(shap_selected), axis=0),
                'Std_SHAP': np.std(shap_selected, axis=0),
                'Min_SHAP': np.min(shap_selected, axis=0),
                'Max_SHAP': np.max(shap_selected, axis=0),
                'Importance': sel_importances
            })
            shap_summary.to_excel(writer, sheet_name='SHAP_Summary', index=False)

            var_info = pd.DataFrame({
                'Variable': valid_vars,
                'Label': [VAR_LABELS.get(v, v) for v in valid_vars],
                'Category': [var_categories.get(v, 'other') for v in valid_vars],
                'Importance': importances,
                'Rank': [results['importance_ranking']['all_variables'][i]['rank'] for i in range(len(valid_vars))]
            })
            var_info.to_excel(writer, sheet_name='Variable_Info', index=False)
    except Exception:
        pass

    plt.rcParams['font.family'] = 'DejaVu Sans'
    plt.rcParams['font.size'] = 10

    fixed_labels = [fix_co2_label(lb) for lb in sel_labels]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6, 8),
                                   gridspec_kw={'width_ratios': [1, 1], 'wspace': 0.05})

    y_positions = np.arange(n_top)[::-1]
    violin_width = 0.8
    all_scatter_x = []
    all_scatter_y = []
    all_scatter_c = []

    np.random.seed(42)

    def get_violin_boundary(shap_vals, violin_width=0.8, resolution=100):
        from scipy import stats as sp_stats
        kde = sp_stats.gaussian_kde(shap_vals, bw_method=0.3)
        x_min, x_max = shap_vals.min(), shap_vals.max()
        x_range = x_max - x_min if x_max > x_min else 1.0
        x_grid = np.linspace(x_min - x_range * 0.1, x_max + x_range * 0.1, resolution)
        density = kde(x_grid)
        max_density = density.max() if density.max() > 0 else 1.0
        normalized_density = density / max_density * (violin_width / 2)
        return x_grid, normalized_density

    for i in range(n_top):
        y_pos = y_positions[i]
        shap_vals = shap_selected[:, i]
        feature_vals = X_selected[:, i]

        violin_parts = ax1.violinplot([shap_vals], positions=[y_pos], widths=[violin_width],
                                      vert=False, showmeans=False, showmedians=False,
                                      showextrema=False, bw_method=0.3)
        for pc in violin_parts['bodies']:
            pc.set_facecolor(colors[i])
            pc.set_alpha(0.3)
            pc.set_edgecolor('#666666')
            pc.set_linewidth(1.0)

        normalized_vals = (feature_vals - feature_vals.min()) / (feature_vals.max() - feature_vals.min() + 1e-8)

        sample_indices = np.random.choice(len(shap_vals), min(120, len(shap_vals)), replace=False)
        shap_sample = shap_vals[sample_indices]
        norm_sample = normalized_vals[sample_indices]

        x_grid, density_half_width = get_violin_boundary(shap_vals, violin_width)

        y_jitter_list = []
        for shap_val in shap_sample:
            closest_idx = np.argmin(np.abs(x_grid - shap_val))
            max_jitter = density_half_width[closest_idx] * 0.85
            jitter = np.random.uniform(-max_jitter, max_jitter) if max_jitter > 0.01 else np.random.uniform(-0.05, 0.05)
            y_jitter_list.append(jitter)

        y_jitter = np.array(y_jitter_list)
        all_scatter_x.extend(shap_sample)
        all_scatter_y.extend(y_pos + y_jitter)
        all_scatter_c.extend(norm_sample)

    scatter = ax1.scatter(all_scatter_x, all_scatter_y, c=all_scatter_c, cmap='coolwarm',
                          s=15, alpha=0.9, edgecolors='white', linewidth=0.3, zorder=5)

    ax1.set_yticks(y_positions)
    ax1.set_yticklabels([fixed_labels[i] for i in range(n_top)], fontsize=14)
    ax1.set_xlabel('SHAP value', fontsize=14, labelpad=10)
    ax1.axvline(x=0, color='black', linestyle='-', alpha=0.7, linewidth=1.5)
    ax1.grid(axis='x', alpha=0.3, linestyle='--')
    ax1.set_axisbelow(True)

    for spine in ax1.spines.values():
        spine.set_visible(True)
        spine.set_color('#555555')

    ax1.set_xlim(-0.13, 0.13)
    ax1.set_xticks([-0.1, 0, 0.1])
    ax1.set_xticklabels(['-0.1', '', '0.1'])
    ax1.tick_params(axis='x', labelsize=12, pad=10)
    ax1.tick_params(axis='y', labelsize=14, pad=14)

    cax = inset_axes(ax1, width="3%", height="25%", loc='lower left',
                     bbox_to_anchor=(0.02, 0.02, 1, 1), bbox_transform=ax1.transAxes)
    cbar = plt.colorbar(scatter, cax=cax, orientation='vertical')
    cbar.ax.yaxis.set_label_coords(3, 0.5)
    cbar.set_label('Feature value', fontsize=11)
    cbar.set_ticks([0, 1])
    cbar.set_ticklabels(['Low', 'High'])
    cbar.ax.tick_params(labelsize=11)

    bar_height = 0.7
    bars = ax2.barh(y_positions, sel_importances, height=bar_height,
                    color=colors, alpha=0.7, edgecolor='white', linewidth=1)

    max_importance = sel_importances.max() if len(sel_importances) > 0 else 0.20
    ax2.set_xlim(0, 0.23)
    ax2.set_xticks([0.05, 0.10, 0.15, 0.20])
    ax2.set_xticklabels(['0.05', '0.10', '0.15', '0.20'])
    ax2.tick_params(axis='x', labelsize=12, pad=10)
    ax2.set_xlabel('Importance', fontsize=14, labelpad=10)

    for bar, pdp_result in zip(bars, pdp_results):
        bar_width = bar.get_width()
        bar_y = bar.get_y()
        bar_height_actual = bar.get_height()

        try:
            pdp_values = pdp_result['average'][0]
            grid_values = pdp_result['grid_values'][0]
        except Exception:
            pdp_values = pdp_result.average[0]
            grid_values = pdp_result.grid_values[0]

        pdp_normalized = (pdp_values - pdp_values.min()) / (pdp_values.max() - pdp_values.min() + 1e-8)
        pdp_scaled = pdp_normalized * bar_height_actual * 0.6

        x_normalized = (grid_values - grid_values.min()) / (grid_values.max() - grid_values.min() + 1e-8)
        x_scaled = x_normalized * bar_width * 0.85

        y_offset = bar_y + bar_height_actual / 2 - pdp_scaled.mean()
        y_scaled = pdp_scaled + y_offset

        ax2.plot(x_scaled, y_scaled, color='black', linewidth=1.8, alpha=0.8)

    ax2.set_yticks(y_positions)
    ax2.set_yticklabels([])

    for spine in ax2.spines.values():
        spine.set_visible(True)
        spine.set_color('#2c3e50')

    for imp, bar in zip(sel_importances, bars):
        ax2.text(bar.get_width() + max_importance * 0.03,
                 bar.get_y() + bar.get_height() / 2,
                 f'{imp:.3f}', va='center', ha='left', fontsize=11)

    ax2.grid(axis='x', alpha=0.3, linestyle='--')
    ax2.set_axisbelow(True)

    plt.tight_layout()
    save_path_png = f'{OUTPUT_DIR}\\shap_importance_filtered_{output_suffix}_spring_rd.png'
    plt.savefig(save_path_png, dpi=600, bbox_inches='tight', pad_inches=0.2)
    plt.show()

    return results


if __name__ == "__main__":
    ensure_output_dir(OUTPUT_DIR)

    SUFFIX = 'Temperate'

    vars_after_step1 = step1_hexbin_plot(INITIAL_VARS, output_suffix=SUFFIX)
    vars_after_step2 = step2_hierarchical_clustering(vars_after_step1, output_suffix=SUFFIX)
    results = step3_shap_analysis_and_plot(vars_after_step2, output_suffix=SUFFIX)
